# 07 · SEI 상(相) 매핑 — halo·ring·spot 픽셀 분류 → phase map + 두께

흐름: **입력 → 전처리 → 개요(median/MAX/center) → radial(예상 peak) → Halo → Ring → Spot → Phase map(확정/예상/약함) → 두께**.
각 픽셀에서 radial integration을 해 halo(비정질)·ring(다결정)·spot(단결정)을 보고, 물질과 상을 **확정/예상/약함**으로 판정한다.
로직은 패키지(`fds.classify_pixels` 등), 이 노트북은 **각 셀 파라미터를 노출**해 데이터마다 조절한다.
플롯 텍스트는 ASCII(한글은 마크다운/print만). 판정: **확정**=스팟 인덱싱(격자 자기일관) / **예상**=링 지문 일치 / **약함**=물질이나 상 불명.

## 1) 입력 — 로드 (경로만 바꾸면 어느 데이터든)

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"   # ★데이터셋
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)
DET_BIN    = 1                       # 검출기 비닝(메모리). q_max는 안 변함
Q_UNIT_HINT= "1/nm"                 # dm 단위(0.043888 1/nm)
CANDIDATES = ["LiF","Li2O","Li3N","Li2CO3","Li2S"]
N_JOBS     = -1                      # 병렬 코어(-1=전부; 32코어면 32)
PHASE_COL  = {"LiF":"#2ca02c","Li2O":"#1f77b4","Li3N":"#9467bd","Li2CO3":"#ff7f0e","Li2S":"#8c564b"}

def _synth(Sy=22,Sx=30,H=88,W=88,seed=0):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/8); halo=lambda r0,s=4:np.exp(-(rr-r0)**2/(2*s**2))
    def sp(r0,n=6,a=4):
        im=np.zeros((H,W))
        for k in range(n):
            t=2*np.pi*k/n; im+=a*np.exp(-((xx-cx-r0*np.cos(t))**2+(yy-cy-r0*np.sin(t))**2)/(2*1.6**2))
        return im
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            base=0.9*beam if iy>=Sy-3 else (beam+sp(20)+0.5*halo(20) if ix<Sx//2 else beam+1.2*halo(18))
            cube[iy,ix]=np.clip(base+0.15*rng.standard_normal((H,W)),0,None)
    return cube
if USE_SYNTHETIC: cube=fds.from_array(_synth(),q_per_px=0.02,name="synthetic")
else:
    cube=fds.load(DM4_PATH,Q_UNIT_HINT)
    if DET_BIN>1: cube=fds.bin_cube_detector(cube,DET_BIN)
scan=cube.scan_shape; QPP=cube.calibration.q_per_px
NAME=("synthetic" if USE_SYNTHETIC else os.path.splitext(os.path.basename(DM4_PATH))[0])
SAVE_DIR=("nb7_outputs" if USE_SYNTHETIC else os.path.dirname(DM4_PATH)+"/nb7_outputs"); os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,n): p=os.path.join(SAVE_DIR,f"{NAME}_{n}.png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(n,h,rows):
    import csv; p=os.path.join(SAVE_DIR,f"{NAME}_{n}.csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(h); w.writerows(rows)
    print("saved:",p)
print("cube:",cube.shape,"| q_per_px=",QPP,"| cores:",os.cpu_count(),"->",os.path.abspath(SAVE_DIR))

## 2) 전처리 — 진단(측정 먼저, 필요할 때만 교정)

인공물(중심/wander/타원/defect)은 교정, 신호 평활(블러)은 금지. 여기선 중심과 상태를 진단한다.

In [ ]:
# --- 이 셀 파라미터 ---
HOT_THRESHOLD  = 8.0    # hot/dead 검출 민감도(작을수록 민감). 진짜 스팟이 지워지면 키우기
WANDER_WARN_PX = 1.0    # 이보다 크면 per-position 정렬 권고
ELLIP_WARN     = 0.02   # 타원율 이보다 크면 타원 보정 권고
diag=fds.diagnose_cube(cube,hot_threshold=HOT_THRESHOLD); center=diag["center"]
print("=== 전처리 진단 ===")
print(f"  center           = ({center[0]:.1f},{center[1]:.1f})  (hot-pixel 제거 후 무게중심)")
print(f"  beam wander      = {diag['wander_px']:.2f} px  (>{WANDER_WARN_PX} 이면 정렬 고려)")
print(f"  detector defects = {100*diag['bad_pixel_frac']:.2f} %")
print(f"  ring ellipticity = {100*diag['ellipticity']:.1f} % @ {diag['ellipse_angle_deg']:.0f}deg  (>{100*ELLIP_WARN:.0f}% 면 보정)")
for n in diag["notes"]: print("  -",n)

## 3) 개요 — median / MAX NBD / center

median=비정질 halo가 잘 보임, MAX=다결정 링·스팟이 모여 보임.

In [ ]:
# --- 이 셀 파라미터 ---
CENTER = None       # None=진단값. 수동이면 (cx,cy)
if CENTER is not None: center=CENTER
med=fds.median_pattern(cube); mx=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
fig,ax=plt.subplots(1,2,figsize=(9,4.4))
ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(*center,"c+",ms=9); ax[0].set_title("median NBD (log) - amorphous halo"); ax[0].axis("off")
ax[1].imshow((np.clip(mx,0,None)/mx.max())**0.3,cmap="magma"); ax[1].plot(*center,"c+",ms=9); ax[1].set_title("MAX NBD (gamma) - rings/spots"); ax[1].axis("off")
plt.tight_layout(); save(fig,"03_overview"); plt.show()

## 4) Radial integration — **실제 halo 봉우리 검출**(q·d) + 예상 링 위치

전체 평균 NBD 방위각 적분에서 **halo 봉우리를 데이터에서 직접 검출**. ★steep 빔꼬리에서는 2·3차 halo가 **봉우리가 아니라 어깨(shoulder)**라
단순 peak로는 FSDP 하나만 잡힘 → **convex(멱함수) 배경을 빼고 contrast로 검출**(어깨도 봉우리로, 약한 고-q halo도 FSDP에 안 묻힘).
오른쪽 그림이 그 검출신호 — 여기서 실제로 뭐가 잡혔는지 눈으로 확인하고, 다르면 **HALO_QS_MANUAL**로 직접 지정. 모든 halo는 **q와 d 둘 다** 표기.

In [ ]:
# --- 이 셀 파라미터 ---
RING_QBEAM=0.20; RING_QMAX=1.0; RING_NSIG=1.5; RING_TOPN=8   # 링 검출
HALO_QLO=0.20               # halo 검출 하한(1/A) — 빔/빔가장자리 배제(★FSDP가 빔쪽이면 낮추기)
HALO_MAXPK=5                # 검출할 halo 최대 개수
HALO_PROM=0.05              # 검출신호(contrast) 대비 최소 prominence. ★약한 halo 안 잡히면 낮추기(0.03), 가짜 많으면 키우기
HALO_MINWIDTH=0.04          # halo=넓은 봉우리만(반폭 1/A). sharp 결정질 링은 배제(→ §6 ring)
HALO_DEG=2                  # convex 배경 차수(2 권장). None이면 rolling-min(어깨 못 잡음)
HALO_QS_MANUAL=None         # ★검출신호 그림 보고 halo를 직접 지정(예: [0.25,0.55,0.80]). None=자동
meanpat=fds.to_pattern(cube)                                    # mean(전체) — halo가 median보다 잘 남음
qd,Id=fds.azimuthal_integrate(meanpat,center,q_per_px=QPP)     # mean -> halo
qm,Im=fds.azimuthal_integrate(mx,center,q_per_px=QPP)          # max -> rings
# halo 봉우리: convex 배경 뺀 contrast에서 검출(어깨 포함). 검출신호(dsig)도 받아서 그림에 표시.
auto_peaks,(dq_,dsig)=fds.amorphous_halo_peaks(qd,Id,q_lo=HALO_QLO,q_hi=RING_QMAX,smooth=3,prominence_frac=HALO_PROM,min_width_q=HALO_MINWIDTH,deg=HALO_DEG,max_peaks=HALO_MAXPK,return_profile=True)
if HALO_QS_MANUAL: halo_peaks=sorted(float(x) for x in HALO_QS_MANUAL)
else: halo_peaks=sorted(auto_peaks) if auto_peaks else [0.25]
halo_q=halo_peaks[0]                                           # FSDP = 최저 q halo(정의)
rings_q=fds.detect_rings(mx,center,QPP,q_beam=RING_QBEAM,q_max=RING_QMAX,nsig=RING_NSIG,top_n=RING_TOPN)
print(f"자동검출 halo: "+", ".join(f"q={p:.3f}(d={1/p:.2f}A)" for p in sorted(auto_peaks)) if auto_peaks else "자동검출 halo: 없음")
print(f"사용 halo 봉우리: "+", ".join(f"q={p:.3f}(d={1/p:.2f}A)" for p in halo_peaks)+(" [수동지정]" if HALO_QS_MANUAL else ""))
print(f"  FSDP = q{halo_q:.3f} (d={1/halo_q:.2f}A) | 검출 링 q="+", ".join(f"{r:.3f}(d{1/r:.2f})" for r in rings_q))
fig,ax=plt.subplots(1,2,figsize=(15,4.3))
# 왼쪽: I(q) + 검출 halo(q,d) + 예상 링
ax[0].semilogy(qm,np.clip(Im,1e-2,None),"k-",lw=0.9,label="MAX I(q)")
ax[0].semilogy(qd,np.clip(Id,1e-2,None),"0.5",lw=0.8,label="mean I(q)")
for c in CANDIDATES:
    for dd,w in fds.COMPOUND_RINGS[c]: ax[0].axvline(1/dd,color=PHASE_COL[c],ls="--",lw=0.4+0.9*w,alpha=0.5)
for r in rings_q: ax[0].axvline(r,color="k",ls=":",lw=0.7)
for p in halo_peaks:
    ax[0].axvline(p,color="r",ls="-",lw=1.3,alpha=0.85)
    ax[0].annotate(f"q={p:.2f}\nd={1/p:.2f}A",(p,Id[np.argmin(abs(qd-p))]),color="r",fontsize=7,ha="center",va="bottom")
import matplotlib.patches as mp
ax[0].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES]+
          [plt.Line2D([],[],color="k",ls=":",label="detected ring"),plt.Line2D([],[],color="r",label="halo (q,d)")],fontsize=7,ncol=2)
ax[0].set_xlabel("q (1/A)"); ax[0].set_ylabel("I(q)"); ax[0].set_title("radial: detected halos (red, q&d) + candidate rings (dashed)")
# 오른쪽: 검출신호(convex 배경 뺀 contrast) — 여기서 봉우리가 실제로 솟는지 확인
ax[1].plot(dq_,dsig,"b-",lw=1.2,label="detection signal (contrast, halo=bump)")
ax[1].axhline(0,color="0.7",lw=0.6)
for p in halo_peaks:
    ax[1].axvline(p,color="r",ls="-",lw=1.1,alpha=0.85)
    ax[1].annotate(f"q={p:.2f}\nd={1/p:.2f}A",(p,dsig[np.argmin(abs(dq_-p))]),color="r",fontsize=7,ha="center",va="bottom")
ax[1].set_xlim(HALO_QLO-0.02,RING_QMAX); ax[1].set_xlabel("q (1/A)"); ax[1].set_ylabel("contrast")
ax[1].set_title("halo detection signal: real halo = clear bump (check here, tune HALO_QS_MANUAL)"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"04_radial_expected"); plt.show()

## 5) Halo 분석 — 물질/진공 판정 + 구조적 halo (진공 대비 cutoff)

**§4에서 검출한 실제 halo 봉우리**들을 프로브로 쓴다(고정 0.3/0.4/0.8 아님). 두 가지 맵:
- **PLAIN**(halo 밴드 세기): 물질/진공 판정용. **세기 ~ 두께**라 두꺼운 곳이 그냥 밝음(구조 아님).
- **STRUCTURAL**(halo *봉우리*): 각 픽셀의 radial을 **매끄러운 convex 배경**(빔꼬리+작은각; halo 밴드는 빼고 log-log
  power-law로 fit)으로 빼서 봉우리를 남긴다. ★직선 flank는 저-q 볼록 빔꼬리에서 **넘겨빼기(overshoot)** 해서 봉우리를 0으로
  죽였음(그래서 그래프가 나빴다) → **convex 배경**으로 교체. 게다가 봉우리 세기는 두께에 비례하므로,
  **contrast = 봉우리/배경**(두께 무관)을 진공 대비 z-score(≈4σ) → 두껍기만 한 featureless는 배제, **진짜 질서**만 밝음.
  넓은 halo도 통째로(낮고 높은 어깨까지) 배경 위로 올라오므로 자연히 잡힌다.
halo는 비정질이라 상 이름은 안 붙임(2Å 축퇴). 어떤 물질인지는 §6 ring/§7 spot이 답.

In [ ]:
# --- 이 셀 파라미터 ---
VAC_PCTL      = 15          # 엄격 기준진공 = 총세기 하위 X% (detector cutoff 기준). 물질이 화면을 많이 채우면 낮추기
HALO_BAND     = (0.15,0.60) # 물질 판정용 broad halo 밴드(1/A) — 이 구간 산란이 진공 대비 유의하면 물질
HALO_SIGMA    = 3.0         # 진공 대비 이 σ 이상 = 진짜 물질(cutoff). ★물질이 과하게 잡히면 키우기(4,5)
HALO_STRONG_S = 8.0         # 이 σ 이상 = 확정물질(강 halo), 그 사이는 예상물질
HALO_DQ       = 0.04        # 개별 halo 반경 PLAIN detector 반폭(1/A)
# STRUCTURAL: convex 배경(power-law) + contrast. §4 검출 봉우리를 프로브로 사용.
STRUCT_DQ     = 0.06        # halo 밴드 반폭(1/A) — 봉우리 폭에 맞춰(넓은 halo면 키우기)
STRUCT_BEAMCUT= 0.14        # 배경 fit 하한(1/A) — 이 아래 빔코어는 fit에서 제외
STRUCT_QMAX   = 1.05        # 배경 fit 상한(1/A)
STRUCT_DEG    = 2           # log-log 배경 다항 차수(2~3; power-law+상수 배경). 너무 크면 봉우리 흡수
STRUCT_CSIG   = 4.0         # contrast(봉우리/배경) 진공 대비 이 σ 이상 = 진짜 질서. ★과검출이면 키우기
NBIN          = 200
# 1) 픽셀 radial stack 1회 + 엄격 진공
q_stack,prof=fds.radial_stack(cube,center,QPP,q_max=1.2,nbin=NBIN,n_jobs=N_JOBS)
vac_ref=fds.strict_vacuum_mask(cube,center=center,q_per_px=QPP,pctl=VAC_PCTL)
# 2) 물질 판정 = broad halo 밴드 전체를 진공 대비(plain 환형; 정확한 peak 불필요, robust)
q0b=0.5*(HALO_BAND[0]+HALO_BAND[1]); dqb=0.5*(HALO_BAND[1]-HALO_BAND[0])
halo_sig,_=fds.detector_map(cube,center,QPP,q0b,dq=dqb,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))
material=halo_sig>HALO_SIGMA
halo_tier=np.where(~material,0,np.where(halo_sig>=HALO_STRONG_S,3,2))
print(f"물질 {100*material.mean():.0f}% (halo밴드 {HALO_BAND}>{HALO_SIGMA}σ) | 확정물질(>{HALO_STRONG_S}σ) {int((halo_tier==3).sum())}px | 예상물질 {int((halo_tier==2).sum())}px | 진공 {int((~material).sum())}px")
med_rad=prof[material.ravel()].mean(0) if material.any() else prof.mean(0)   # 물질 평균 radial
# 3) halo 프로브 = §4에서 검출한 실제 봉우리(고정값 아님). FSDP = 가장 강한 봉우리(§4 halo_q).
halo_probes=[round(p,3) for p in halo_peaks if STRUCT_BEAMCUT<p<STRUCT_QMAX-0.02]
if not halo_probes: halo_probes=[round(halo_q,3)]
fsdp_q=min(halo_probes,key=lambda p:abs(p-halo_q))            # 표시용 FSDP = 강한 봉우리에 가장 가까운 프로브
# PLAIN(세기=두께) & STRUCTURAL(contrast=봉우리/배경, convex 배경) 맵
plain_maps=[fds.detector_map(cube,center,QPP,p,dq=HALO_DQ,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))[0] for p in halo_probes]
_bump=fds.halo_bump_maps(cube,center,QPP,halo_probes,dq=STRUCT_DQ,beam_cut=STRUCT_BEAMCUT,q_max=STRUCT_QMAX,deg=STRUCT_DEG,vacuum_mask=vac_ref,_stack=(q_stack,prof))
struct_maps=[_bump[p]["csig"] for p in halo_probes]        # contrast 유의도(두께 무관)
struct_raw =[_bump[p]["raw"]  for p in halo_probes]         # 봉우리 세기(그림용)
# 대표로 보여줄 halo = STRUCTURAL이 가장 강한 halo(FSDP일 수도, 2·3차일 수도 — 데이터가 정함)
struct_frac=[float((sg>STRUCT_CSIG).mean()) for sg in struct_maps]
i_show=int(np.argmax(struct_frac)); q_show=halo_probes[i_show]; struct_sig=struct_maps[i_show]
print(f"검출 halo q ={[round(p,3) for p in halo_probes]}  d(A)={[round(1/p,2) for p in halo_probes]}  (FSDP={fsdp_q:.3f})")
print("  PLAIN  유의(>3σ) % (세기=두께):    ",[round(100*(sg>HALO_SIGMA).mean(),1) for sg in plain_maps])
print(f"  STRUCT 유의(>{STRUCT_CSIG}σ contrast) % (질서): ",[round(100*f,1) for f in struct_frac]," ← 두께무관, halo별로 다 봄")
print(f"  → STRUCTURAL 최강 halo = q{q_show:.3f} (d={1/q_show:.2f}A), {100*struct_frac[i_show]:.1f}%")
# --- 그림 A: 대표 halo NBD(검출 halo 전부 원) | PLAIN | STRUCTURAL(최강 halo) | 물질 마스크 ---
iy,ix=np.unravel_index(int(np.argmax(np.where(material,halo_sig,-np.inf))),scan)
fig,ax=plt.subplots(1,4,figsize=(18,4.2))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="magma"); ax[0].plot(*center,"c+",ms=8)
_hc=plt.cm.cool(np.linspace(0,1,len(halo_probes)))
for p,col in zip(halo_probes,_hc): ax[0].add_patch(plt.Circle(center,p/QPP,fill=False,ec=col,ls="--",lw=1.0))
ax[0].set_title(f"representative material pixel ({iy},{ix})\nall detected halos: {[round(p,2) for p in halo_probes]}",fontsize=9); ax[0].axis("off")
im=ax[1].imshow(np.where(material,halo_sig,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("PLAIN halo-band (intensity ~ thickness)"); ax[1].axis("off")
im=ax[2].imshow(np.where(material,struct_sig,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[2],fraction=0.046)
ax[2].set_title(f"STRUCTURAL: strongest halo q={q_show:.2f} (d={1/q_show:.2f}A)\n(contrast=bump/bg, thickness-free)",fontsize=9); ax[2].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
cmH=ListedColormap(["#000000","#4575b4","#2ca02c"]); nmH=BoundaryNorm([-.5,.5,2.5,3.5],3)
ax[3].imshow(halo_tier,cmap=cmH,norm=nmH); ax[3].set_title(f"material (>{HALO_SIGMA}s): vacuum/weak/strong"); ax[3].axis("off")
plt.tight_layout(); save(fig,"05_halo_material"); plt.show()
# --- 그림 B: 각 halo 반경 PLAIN(위) vs STRUCTURAL(아래) — 0.3/0.4/0.8이 진짜 봉우리인지 ---
nH=len(halo_probes)
fig,ax=plt.subplots(2,nH,figsize=(3.4*nH,7.0),squeeze=False)
for j,(p,pl,st) in enumerate(zip(halo_probes,plain_maps,struct_maps)):
    im=ax[0,j].imshow(np.where(material,pl,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[0,j],fraction=0.046)
    ax[0,j].set_title(f"PLAIN q={p:.2f} (d={1/p:.2f}A)",fontsize=9); ax[0,j].axis("off")
    im=ax[1,j].imshow(np.where(material,st,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[1,j],fraction=0.046)
    ax[1,j].set_title(f"STRUCTURAL q={p:.2f} (contrast sig)",fontsize=9); ax[1,j].axis("off")
fig.suptitle("per-halo-radius: PLAIN (intensity/thickness) vs STRUCTURAL (contrast=bump/bg, thickness-free)")
plt.tight_layout(); save(fig,"05_halo_peaks"); plt.show()
# --- 그림 C: convex 배경 위 봉우리 = STRUCTURAL. 물질평균 radial에 매끄러운 배경(빔꼬리+작은각)을
#     halo 밴드 빼고 power-law로 fit해 그리고, 그 위로 올라온 봉우리(주황)를 §4 검출 봉우리마다 칠한다.
#     ★직선 flank가 아니라 convex 배경 → 저-q 볼록 꼬리에서도 봉우리가 배경 위로 제대로 올라옴(그래프가 정상).
vac_rad=prof[vac_ref.ravel()].mean(0)
bg1d=fds.halo_background_1d(q_stack,med_rad,halo_probes,dq=STRUCT_DQ,beam_cut=STRUCT_BEAMCUT,q_max=STRUCT_QMAX,deg=STRUCT_DEG)
bump1d=np.clip(med_rad-bg1d,0,None)
vrng=(q_stack>=STRUCT_BEAMCUT)&(q_stack<=STRUCT_QMAX)
fig,ax=plt.subplots(1,2,figsize=(15,4.4))
# 패널0: 전체 radial + convex 배경 + 봉우리(주황) + 검출 봉우리 위치
ax[0].semilogy(q_stack,np.clip(med_rad,1e-3,None),"k-",lw=1.3,label="material mean I(q)")
ax[0].semilogy(q_stack,np.clip(bg1d,1e-3,None),"r--",lw=1.2,label="convex background (power-law)")
ax[0].semilogy(q_stack,np.clip(vac_rad,1e-3,None),"0.6",lw=0.9,label="vacuum mean")
for j,p in enumerate(halo_probes):
    ax[0].axvline(p,color="tab:orange",ls=":",lw=1.0)
    ax[0].annotate(f"{p:.2f}",(p,med_rad[np.argmin(abs(q_stack-p))]),fontsize=7,color="tab:orange",ha="center",va="bottom")
ax[0].set_xlim(0.10,1.1); ax[0].set_xlabel("q (1/A)"); ax[0].set_ylabel("I(q)")
ax[0].set_title("material mean + convex background (halo bands excised from fit)"); ax[0].legend(fontsize=8)
# 패널1: 봉우리(배경 뺀 것) linear — 검출 봉우리에서 실제로 봉우리가 솟는다
ax[1].plot(q_stack[vrng],bump1d[vrng],"k-",lw=1.3,label="material mean - background")
ax[1].fill_between(q_stack[vrng],0,bump1d[vrng],color="tab:orange",alpha=0.5)
ax[1].axhline(0,color="0.7",lw=0.6)
for p in halo_probes:
    b=(q_stack>=p-STRUCT_DQ)&(q_stack<=p+STRUCT_DQ)
    ax[1].axvspan(p-STRUCT_DQ,p+STRUCT_DQ,color="tab:orange",alpha=0.15)
    ax[1].axvline(p,color="tab:orange",ls=":",lw=1.0)
    ax[1].annotate(f"q={p:.2f}\nd={1/p:.2f}A",(p,bump1d[b].max() if b.any() else 0),fontsize=7,ha="center",va="bottom")
ax[1].set_xlim(STRUCT_BEAMCUT,STRUCT_QMAX); ax[1].set_xlabel("q (1/A)")
ax[1].set_title("bump above background = STRUCTURAL halo (peaks at DETECTED q, flat elsewhere)"); ax[1].legend(fontsize=8)
fig.suptitle("STRUCTURAL halo = bump above the smooth convex background (not a straight-chord flank)")
plt.tight_layout(); save(fig,"05_halo_profile"); plt.show()
# --- 그림 D: 각 halo 반경에서 '강한 신호 픽셀'의 평균 NBD — 뿌연 링(halo)인가 스팟(결정질)인가 ---
STRONG_PCTL=90   # 각 반경 STRUCTURAL 맵 상위 %를 '강한 신호'로 골라 그 픽셀들의 NBD 평균
fig,ax=plt.subplots(1,nH,figsize=(3.6*nH,3.8)); ax=np.atleast_1d(ax)
for a,p,st in zip(ax,halo_probes,struct_maps):
    thr=np.nanpercentile(st[material],STRONG_PCTL) if material.any() else np.inf
    strong=material&(st>=thr)&(st>0)
    n=int(strong.sum())
    if n>=3:
        mp=fds.average_pattern(cube,strong)
        a.imshow((np.clip(mp,0,None)/np.max(mp))**0.3,cmap="magma"); a.plot(*center,"c+",ms=7)
        a.add_patch(plt.Circle(center,p/QPP,fill=False,ec="cyan",ls="--",lw=1.0))
        a.set_title(f"q={p:.2f} (d={1/p:.2f}A)\nmean NBD of {n} strong px",fontsize=9)
    else:
        a.set_title(f"q={p:.2f}: strong px<3",fontsize=9)
    a.axis("off")
fig.suptitle("mean NBD at STRONG STRUCTURAL pixels per halo radius: diffuse RING=amorphous halo, SPOTS=crystalline")
plt.tight_layout(); save(fig,"05_halo_meanNBD"); plt.show()
save_csv("05_halo",["halo_q","halo_d","plain_frac>3s",f"struct_frac>{STRUCT_CSIG}s(contrast)"],
         [[f"{p:.4f}",f"{1/p:.3f}",f"{(pl>HALO_SIGMA).mean():.4f}",f"{(st>STRUCT_CSIG).mean():.4f}"] for p,pl,st in zip(halo_probes,plain_maps,struct_maps)]
         +[["material_band","",f"{material.mean():.4f}",""]])

## 6) Ring 분석 — MAX에서 검출한 링마다 virtual image + 영역 NBD + flank 검증 (§5처럼)

§5(halo)와 같은 흐름을 **sharp 링**에 적용. 비정질 halo와 달리 링은 **MAX NBD**에 또렷하다 → **MAX에서 링(4~5개)을 검출**해 위치로 삼음.
각 링마다:
- **PLAIN** 환형 detector virtual image = 그 반경 세기 분포(∝두께 포함, 어디에 신호가 있나)
- **STRUCTURAL**(flank-빼기) virtual image = **비정질 배경 위로 솟는 sharp 링만**(broad halo는 flank로 사라짐). 비정질(물질) 대비 z-score.
- STRUCTURAL이 **뭉친 영역(cluster)**을 찾아 그 영역의 **평균 NBD** → 또렷한 링/스팟(결정질)인지 확인(여러 군데면 각각).
- **flank 검증 그래프**(§5처럼 radial): 그 영역 평균 radial에서 링이 배경 위 봉우리인지 **직선 flank**로 확인(링=sharp라 직선 flank가 맞다; halo=convex와 반대).
여기까진 **신호 우선**(진짜 결정질 영역이 어디인가). **어떤 상**인지는 §7 인덱싱이 확정.

In [ ]:
# --- 이 셀 파라미터 ---
RING_DQ       = 0.03   # 링 detector 반폭(1/A) ~ 링 폭
RING_FLANK    = 2.0    # flank 배수(sharp 링 배경빼기; halo와 달리 링엔 직선 flank가 맞다)
RING_SIGMA    = 4.0    # 비정질 배경 대비 이 σ 이상 = 진짜 sharp 링. ★잡링 많으면 키우기(5,6)
RING_MINCLUS  = 3      # 최소 cluster 크기(px) — 이보다 작은 점은 노이즈로 버림
RING_QLO0     = 0.24   # 링으로 쓸 하한(1/A, FSDP 아래 제외)
RING_HALOGAP  = 0.035  # 검출 halo 봉우리에서 이만큼 안쪽이면 '링 아님(halo)'으로 제외(1/A)
RING_MAXRINGS = 6      # 최대 링 개수
RING_QS_MANUAL= None   # ★MAX에서 눈으로 본 링 위치를 직접 지정(예: [0.43,0.50,0.70,0.82]). None=자동검출
from scipy.ndimage import label as _label
# 링 위치: 수동 지정 우선, 없으면 §4 MAX 검출 링(rings_q) 중 halo 봉우리와 안 겹치는 sharp 링.
if RING_QS_MANUAL:
    ring_qs=sorted([float(x) for x in RING_QS_MANUAL])
else:
    ring_qs=sorted([qq for qq in rings_q if RING_QLO0<qq<1.12 and all(abs(qq-hp)>RING_HALOGAP for hp in halo_peaks)])[:RING_MAXRINGS]
    print(f"  자동검출 링 q = {[round(r,3) for r in ring_qs]}  (MAX에서 본 링과 다르면 RING_QS_MANUAL로 직접 지정)")
if not ring_qs:
    print("  (MAX에서 halo와 구분되는 sharp 링 없음 → §7 spottiness로 결정질 탐색; RING_QS_MANUAL 지정 가능)")
print(f"=== ring 분석: MAX 검출 링 q(1/A) = {[round(r,3) for r in ring_qs]}  d(A) = {[round(1/r,2) for r in ring_qs]} ===")
def _flankbase(qq,p1d,q0,dq=RING_DQ,flank=RING_FLANK):        # peak_above_flank와 동일한 배경(양옆 flank 평균)
    band=(qq>=q0-dq)&(qq<=q0+dq); lo=(qq>=q0-flank*dq)&(qq<q0-dq); hi=(qq>q0+dq)&(qq<=q0+flank*dq)
    vals=[p1d[lo].mean() if lo.any() else np.nan,p1d[hi].mean() if hi.any() else np.nan]
    return band,lo,hi,float(np.nanmean(vals))
plain_rings=[]; struct_rings=[]; clusters=[]; cryst_region=np.zeros(scan,bool)
for rq in ring_qs:
    psig,_=fds.detector_map(cube,center,QPP,rq,dq=RING_DQ,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))   # PLAIN(세기)
    _,raw=fds.detector_map(cube,center,QPP,rq,dq=RING_DQ,flank=RING_FLANK,vacuum_mask=vac_ref,_stack=(q_stack,prof)) # flank 뺀 높이
    ssig=fds.significance(raw,material)                        # 비정질(물질) 배경 대비 z
    plain_rings.append(psig); struct_rings.append(ssig)
    lab,n=_label(material&(ssig>RING_SIGMA))
    sizes=[(g,int((lab==g).sum())) for g in range(1,n+1)]
    big=sorted([s for s in sizes if s[1]>=RING_MINCLUS],key=lambda s:-s[1])
    clusters.append([lab==g for g,_ in big])
    for m in clusters[-1]: cryst_region|=m
    print(f"  q={rq:.3f} (d={1/rq:.2f}A): STRUCT>{RING_SIGMA}s {int((material&(ssig>RING_SIGMA)).sum())}px, cluster {len(big)}개"+(f" (최대 {big[0][1]}px)" if big else ""))
ring_pred=cryst_region                                        # §7이 쓰는 결정질 후보 영역
print(f"  결정질(링) 영역 합집합 {int(cryst_region.sum())}px / 물질 {int(material.sum())}px")
# --- 그림: 링마다  PLAIN | STRUCTURAL | 최대 cluster 평균 NBD | flank 검증 radial ---
nR=len(ring_qs)
if nR==0: print("  → sharp 결정질 링 없음: §6 그림은 빈 축, §7(spot)에서 spottiness로 결정질 탐색")
fig,ax=plt.subplots(max(nR,1),4,figsize=(17,3.7*max(nR,1)),squeeze=False)
for i,rq in enumerate(ring_qs):
    im=ax[i,0].imshow(np.where(material,plain_rings[i],np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[i,0],fraction=0.046)
    ax[i,0].set_title(f"PLAIN q={rq:.3f} (d={1/rq:.2f}A)\nintensity ~ thickness",fontsize=9); ax[i,0].axis("off")
    im=ax[i,1].imshow(np.where(material,struct_rings[i],np.nan),cmap="inferno",vmin=0); plt.colorbar(im,ax=ax[i,1],fraction=0.046)
    ax[i,1].set_title(f"STRUCTURAL q={rq:.3f} (flank, vs amorphous)",fontsize=9); ax[i,1].axis("off")
    if clusters[i]:
        cm=clusters[i][0]; mp_=fds.average_pattern(cube,cm)
        ax[i,2].imshow((np.clip(mp_,0,None)/np.max(mp_))**0.3,cmap="magma"); ax[i,2].plot(*center,"c+",ms=7)
        ax[i,2].add_patch(plt.Circle(center,rq/QPP,fill=False,ec="cyan",ls="--",lw=1.0))
        ax[i,2].set_title(f"mean NBD of biggest cluster\n({int(cm.sum())}px, {len(clusters[i])} cluster(s))",fontsize=9); ax[i,2].axis("off")
        reg_rad=prof[cm.ravel()].mean(0)                       # 영역 평균 radial
    else:
        ax[i,2].text(0.5,0.5,f"no cluster >= {RING_MINCLUS}px",ha="center",va="center"); ax[i,2].axis("off")
        reg_rad=med_rad
    band,lo,hi,base=_flankbase(q_stack,reg_rad,rq)             # flank 검증(직선/평평 배경)
    win=(q_stack>=rq-3*RING_FLANK*RING_DQ)&(q_stack<=rq+3*RING_FLANK*RING_DQ)
    ax[i,3].plot(q_stack[win],reg_rad[win],"k-",lw=1.3,label="region mean I(q)")
    for mask,c in [(lo,"0.7"),(hi,"0.7")]:
        qs=q_stack[mask]
        if qs.size: ax[i,3].axvspan(qs.min(),qs.max(),color=c,alpha=0.3)
    qb=q_stack[band]; ax[i,3].plot([qb.min(),qb.max()],[base,base],"r--",lw=1.1,label="flank background")
    ax[i,3].fill_between(q_stack,base,reg_rad,where=band&(reg_rad>base),color="tab:orange",alpha=0.6,label="ring bump")
    ax[i,3].axvline(rq,color="r",ls=":",lw=0.8); ax[i,3].set_xlim(win.nonzero()[0].size and q_stack[win].min(),q_stack[win].max())
    bump=float(np.clip(reg_rad[band]-base,0,None).sum())
    ax[i,3].set_title(f"flank check: bump={bump:.0f} above background\n(real ring if bump>0)",fontsize=9)
    ax[i,3].set_xlabel("q (1/A)"); ax[i,3].legend(fontsize=7)
fig.suptitle("per detected ring: PLAIN (thickness) | STRUCTURAL (real ring vs amorphous) | region mean NBD | flank verification",y=1.0)
plt.tight_layout(); save(fig,"06_ring"); plt.show()
save_csv("06_ring",["ring_q","ring_d","struct>sig_px","n_cluster","biggest_px","max_struct_sig"],
         [[f"{rq:.4f}",f"{1/rq:.3f}",int((material&(struct_rings[i]>RING_SIGMA)).sum()),len(clusters[i]),
           int(clusters[i][0].sum()) if clusters[i] else 0,f"{float(np.nanmax(struct_rings[i])):.1f}"] for i,rq in enumerate(ring_qs)])

## 7) Spot 분석 — ring 영역 안에서만 클러스터 인덱싱 (확정)

**ring 예상 영역(그리고 spottiness 높은 곳)만** 대상으로 한다 — 노이즈에서 수천 개 잡던 문제를 없앤다. 그 영역을
**연결성분(grain)으로 묶어** 각 grain의 NBD를 평균내 스팟을 검출·인덱싱: 한 상으로 **≥4개 자기일관(|g|+각도)=확정**,
갈리거나 2개=예상. 대표 spot 픽셀 NBD(검출 스팟), spottiness 맵, grain별 판정 오버레이.

In [ ]:
# --- 이 셀 파라미터 ---
SPOT_REGION_PCTL = 90    # ring 예상이 적을 때 보강: 물질 내 spottiness 상위 %도 결정질 후보에 포함
SPOT_MIN_GRAIN   = 2     # 최소 grain 크기(px)
SPOT_NMAD        = 8.0   # grain 평균 DP 스팟 문턱(median+NMAD*MAD). 스팟 너무 많으면 키우기
from scipy.ndimage import label as _label
spotmap=fds.crystallinity_map(cube,center=center,q_per_px=QPP,n_jobs=N_JOBS)   # 방위각 spottiness
# 결정질 후보 영역 = ring 예상 ∪ (물질 내 spottiness 상위)
region=ring_pred | (material & (spotmap>np.nanpercentile(spotmap[material],SPOT_REGION_PCTL)))
labels,ng=_label(region)
print(f"=== spot: 결정질 영역 {int(region.sum())}px → grain {ng}개 (ring 예상 {int(ring_pred.sum())}px 포함) ===")
spot_tier=np.zeros(scan,int); spot_phase=np.full(scan,-1,int); confirmed=set(); grain_rows=[]
for g in range(1,ng+1):
    m=labels==g
    if m.sum()<SPOT_MIN_GRAIN: continue
    pat=fds.average_pattern(cube,m)
    sp=fds.detect_spots(pat,center,QPP,n_mad=SPOT_NMAD,min_dist=3,tophat=11,q_max=1.15)
    gs=fds.spots_to_gvectors(sp,center,QPP)
    best=None
    if len(gs)>=2: best,_=fds.index_pattern(gs,candidates=CANDIDATES,tol_g=0.03,tol_ang=6.0,min_score=0.6,min_complete=0.6,confirm_min_spots=4)
    if best is None: continue
    ph=best["phase"]; ki=CANDIDATES.index(ph)
    if best["indexed"]: spot_tier[m]=3; spot_phase[m]=ki; confirmed.add(ph); tag="[확정]"
    elif best["n_matched"]>=2: spot_tier[m]=np.maximum(spot_tier[m],2); spot_phase[m]=np.where(spot_phase[m]<0,ki,spot_phase[m]); tag="[예상]"
    else: tag="[약함]"
    grain_rows.append([g,int(m.sum()),len(gs),ph,int(best["indexed"]),str(best["zone"]),best["n_matched"],f"{best['completeness']:.2f}"])
    print(f"  grain{g:2d} {int(m.sum()):3d}px spots{len(gs):2d}: {tag} {ph:6s} zone{str(best['zone']):12s} m{best['n_matched']}/{best['n_total']} comp{best['completeness']:.2f}")
print(f"  => 인덱싱 확정 상: {sorted(confirmed) or '없음'}")
# 대표 spot 픽셀 = spottiness 최대(물질 내)
iy,ix=np.unravel_index(int(np.argmax(np.where(material,spotmap,-np.inf))),scan)
fig,ax=plt.subplots(1,3,figsize=(14,4.3))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
sp=fds.detect_spots(pat,center,QPP,n_mad=SPOT_NMAD,min_dist=3,tophat=11,q_max=1.15)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="gray"); ax[0].plot(*center,"c+",ms=8)
ax[0].scatter([s[0] for s in sp],[s[1] for s in sp],s=30,facecolors="none",edgecolors="yellow",lw=0.8)
ax[0].set_title(f"representative SPOT pixel ({iy},{ix})\n{len(sp)} spots"); ax[0].axis("off")
im=ax[1].imshow(np.where(material,spotmap,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("spottiness (azimuthal variance)"); ax[1].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
ax[2].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
ax[2].imshow(np.where(spot_tier==3,spot_phase,np.nan),cmap=cmP,norm=nmP)                # 확정 grain(상 색)
ax[2].imshow(np.where(spot_tier==2,1.0,np.nan),cmap="autumn",vmin=0,vmax=2,alpha=0.7)   # 예상 grain
ax[2].set_title("spot grains: color=indexed phase, orange=predict"); ax[2].axis("off")
plt.tight_layout(); save(fig,"07_spot"); plt.show()
if grain_rows: save_csv("07_spot_grains",["grain","size","n_spots","phase","indexed","zone","n_matched","completeness"],grain_rows)

## 8) Phase mapping — 확정 / 예상 / 결정질(상불명) / 비정질 (종합)

상 이름은 **spot 인덱싱(§7)만** 붙인다(신호 우선 원칙): **확정**=≥4스팟 자기일관 · **예상**=2스팟.
**결정질(상불명)**=§6 링 신호는 진짜인데 spot 인덱싱이 상을 확정 못 한 영역. **비정질**=halo만(§5). **없음**=진공.
확정 > 예상 > 결정질 > 비정질 우선.

In [ ]:
# 종합 tier: 4 확정(spot) > 3 예상(spot) > 2 결정질-상불명(ring) > 1 비정질(halo) > 0 진공
tier=np.where(~material,0,1)                              # 물질=비정질 기본
phase_idx=np.full(scan,-1,int)
cn=cryst_region&material                                  # §6 링 신호 = 진짜 결정질(상 이름은 아직)
tier[cn]=2
pm=(spot_tier==2)                                         # §7 spot 예상(2스팟) → 상 이름 있음
tier[pm]=np.maximum(tier[pm],3); phase_idx[pm]=spot_phase[pm]
cm=(spot_tier==3)                                         # §7 spot 확정(≥4스팟) — 최우선
tier[cm]=4; phase_idx[cm]=spot_phase[cm]
print("=== 픽셀 종합 판정 ===")
for t,name in [(4,"확정"),(3,"예상"),(2,"결정질(상불명)"),(1,"비정질"),(0,"없음")]: print(f"  {name}: {int((tier==t).sum())} px | ",end="")
print()
for k,c in enumerate(CANDIDATES):
    cf=int(((phase_idx==k)&(tier==4)).sum()); pr=int(((phase_idx==k)&(tier==3)).sum())
    if cf or pr: print(f"    {c:7s}: 확정 {cf} px, 예상 {pr} px")
conf_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((phase_idx==k)&(tier==4)).any()})
pred_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((phase_idx==k)&(tier==3)).any()})
print(f"  => 확정 상: {conf_ph or '없음'} | 예상 상: {pred_ph or '없음'} | 결정질(상불명) {int((tier==2).sum())}px")
from matplotlib.colors import ListedColormap,BoundaryNorm
import matplotlib.patches as mp
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
def layer(t): return np.where((tier==t)&(phase_idx>=0),phase_idx,np.nan)
fig,ax=plt.subplots(1,4,figsize=(19,4.4))
ax[0].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[0].imshow(layer(4),cmap=cmP,norm=nmP)
ax[0].set_title(f"CONFIRMED (spot indexed): {int((tier==4).sum())} px"); ax[0].axis("off")
ax[0].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="lower right",framealpha=.6)
ax[1].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[1].imshow(layer(3),cmap=cmP,norm=nmP)
ax[1].set_title(f"PREDICTED (2-spot): {int((tier==3).sum())} px"); ax[1].axis("off")
ax[2].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[2].imshow(np.where(tier==2,1.0,np.nan),cmap="cool",vmin=0,vmax=1.4)
ax[2].set_title(f"CRYSTALLINE, phase unknown (ring): {int((tier==2).sum())} px"); ax[2].axis("off")
ax[3].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1); ax[3].imshow(np.where(tier==1,1.0,np.nan),cmap="Greys",vmin=0,vmax=1.4)
ax[3].set_title(f"AMORPHOUS (halo only): {int((tier==1).sum())} px"); ax[3].axis("off")
fig.suptitle("PHASE MAP - CONFIRMED / PREDICTED (phase named) | CRYSTALLINE-unknown / AMORPHOUS")
plt.tight_layout(); save(fig,"08_phase_map"); plt.show()
save_csv("08_phase_tiers",["phase","confirmed_px","predicted_px"],
         [[c,int(((phase_idx==CANDIDATES.index(c))&(tier==4)).sum()),int(((phase_idx==CANDIDATES.index(c))&(tier==3)).sum())] for c in CANDIDATES]
         +[["_crystalline_unknown","",int((tier==2).sum())],["_amorphous","",int((tier==1).sum())]])

## 9) 두께 계산 — t/lambda = ln(I_total / I_beam)

앞에서 **물질/진공을 이미 판정**했으니 이를 활용한다. 진공(=물질 없는 곳)을 기준으로 dark를 자기교정하고 영점을 맞춘 **상대 두께**.
낮음=얇음(표면/껍데기), 높음=두꺼움. (수렴빔·각도분리라 절대 nm은 λ 필요 → 상대값만 신뢰.)

In [ ]:
# --- 이 셀 파라미터 ---
BEAM_RADIUS_PX=None    # 직접빔 디스크 반경(px). None=자동(~det/20)
VAC_PCTL=15            # 총세기 하위 X% = 엄격 기준진공(dark/영점용, 확실히 빈 곳). 물질이 많으면 낮추기
_flat=cube._flat_patterns(); _tot=np.asarray(_flat,float).reshape(_flat.shape[0],-1).sum(1)
vac_ref=np.asarray(_tot<=np.percentile(_tot,VAC_PCTL),bool).reshape(scan)   # 엄격 기준진공(오염 방지)
tmap,tex=fds.thickness_map(cube,center=center,beam_radius=BEAM_RADIUS_PX,vacuum_mask=vac_ref,return_extras=True)
tin=tmap[material]; tin=tin[np.isfinite(tin)]; tvac=tmap[vac_ref]; tvac=tvac[np.isfinite(tvac)]
print(f"dark 자기교정 D={tex['dark']:.3g} | 영점 offset={tex['offset']:.3g} | beam_r={tex['beam_radius']:.1f}px")
print(f"진공 t/lambda mean {tvac.mean():+.3f} (≈0 정상) | 물질 t/lambda mean {np.mean(tin):.2f} (5~95%: {np.percentile(tin,5):.2f}~{np.percentile(tin,95):.2f})")
fig,ax=plt.subplots(1,2,figsize=(11,4.2))
im=ax[0].imshow(np.where(material,tmap,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[0].set_title("relative thickness t/lambda (vacuum-referenced)"); ax[0].axis("off")
ax[1].hist(tin,bins=50,color="0.4",label="material"); ax[1].axvline(0,color="r",ls="--",lw=1,label="vacuum (0)")
ax[1].set_xlabel("t/lambda"); ax[1].set_ylabel("# positions"); ax[1].set_title("thickness distribution (thin <-> thick)"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"09_thickness"); plt.show()
save_csv("09_thickness_stats",["metric","value"],
         [["dark",f"{tex['dark']:.4g}"],["vac_mean",f"{tvac.mean():.4f}"],["mat_mean",f"{np.mean(tin):.4f}"],
          ["mat_p05",f"{np.percentile(tin,5):.4f}"],["mat_p95",f"{np.percentile(tin,95):.4f}"],["material_frac",f"{material.mean():.4f}"]])
print("[정리] 확정/예상=스팟 인덱싱된 상 / 결정질(상불명)=링 신호는 진짜지만 상 미확정 / 비정질=halo만. 두께는 진공 대비 상대값.")